## 9. Çoklu Aracı Kullanan Ajan (Multi-Tool Agent)
* Amaç: Ajanların aynı anda birçok tool kullanarak karar vermesini göstermek.
* Senaryo: Kullanıcı bir şehir planı yapar, ajan hava durumu, harita, maliyet gibi kaynaklara başvurur.
* Kazandırılan: Çok modüllü karar süreçleri.
* Kullanılan Bileşenler: ZeroShotAgent, initialize_agent, Multiple Tools

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               "langchain-experimental>=0.0.64" \
               transformers accelerate sentencepiece \
               duckduckgo-search wikipedia arxiv requests

# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

from langchain.agents import initialize_agent, AgentType

# Araçlar
from langchain_community.tools import DuckDuckGoSearchRun, RequestsGetTool
from langchain_experimental.tools.python.tool import PythonREPLTool

from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.tools.arxiv.tool import ArxivQueryRun

# İsteğe bağlı: daha deterministik davranış için uyarılar
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 3) LLM: Açık kaynak Qwen (küçük ve hızlı)
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # Alternatif: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,     # araç seçimi ve muhakeme için daha tutarlı
    top_p=0.9,
    repetition_penalty=1.05,
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) ARAÇLARI OLUŞTURMA
# ============================================================
# 4.1 Web Araması
search_tool = DuckDuckGoSearchRun(name="WebSearch")

# 4.2 Wikipedia
wiki_api = WikipediaAPIWrapper(lang="tr", top_k_results=3, doc_content_chars_max=4000)
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api, name="WikipediaTR")

# 4.3 arXiv (bilimsel makale arama)
arxiv_api = ArxivAPIWrapper(top_k_results=3, doc_content_chars_max=4000)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_api, name="Arxiv")

# 4.4 HTTP GET (ham içerik çekme)
http_get_tool = RequestsGetTool(name="HttpGet")

# 4.5 Python REPL (hesaplama / küçük kodlar)
python_tool = PythonREPLTool(name="Python")

tools = [search_tool, wiki_tool, arxiv_tool, http_get_tool, python_tool]

# ============================================================
# 5) AGENT: ZERO_SHOT_REACT_DESCRIPTION
# ============================================================
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # ReAct tarzı akıl yürütme + araç çağrımı
    verbose=True,                 # eğitimsel: düşünme/ara adımlarını göster
    handle_parsing_errors=True,   # küçük modellerde sağlamlık
)

# ============================================================
# 6) ÇOKLU ARAÇ KULLANIMINI TETİKLEYEN KOMPOZİT SORU
# ============================================================
prompt_multi = """
CRISPR-Cas9 tekniğini 2-3 cümlede Türkçe açıkla (kısa ve teknik).
Ardından arXiv'de son 2 yılda CRISPR ile ilgili iki önemli makale başlığını ve yıllarını bul.
Son olarak Türkiye'de bu deneyin basit set-up maliyetini yaklaşık 25.000 TL varsayalım;
1 USD = 32.5 TL kabulüyle USD karşılığını Python ile hesaplayıp sonucu ver.
Gerek oldukça uygun araçları (Wikipedia, Arxiv, WebSearch, Python) kullan.
"""

print("\n================= SORU 1 (Multi-Tool) =================\n")
ans1 = agent.run(prompt_multi)
print("\n================= CEVAP 1 =================\n")
print(ans1)

# ============================================================
# 7) BAŞKA ÖRNEKLER (farklı araçları tetiklemek için)
# ============================================================

# 7.1 Sadece Wikipedia eğilimli bir soru
prompt_wiki = "Transformer mimarisini kısa anlat; 'Attention is All You Need' çalışmasının kısa adını ve yılını belirt."
print("\n================= SORU 2 (Wikipedia +/ veya WebSearch) =================\n")
ans2 = agent.run(prompt_wiki)
print("\n================= CEVAP 2 =================\n")
print(ans2)

# 7.2 Web araması ve HTTP GET: belirli bir sayfayı bulup çekme (örnek)
prompt_http = """
LangChain'in GitHub sayfasını bul ve HTTP GET ile sayfa başlığını çekip özetle.
Sadece başlığa ve kısa açıklamaya yer ver.
"""
print("\n================= SORU 3 (WebSearch + HttpGet) =================\n")
ans3 = agent.run(prompt_http)
print("\n================= CEVAP 3 =================\n")
print(ans3)

# 7.3 Python hesaplama: birim dönüşümü/hesap (ajanın Python aracı kullanması beklenir)
prompt_math = """
Bir aracın menzili 420 kilometre ise mil cinsinden yaklaşık değeri nedir?
1 mil = 1.60934 km kabul et ve Python ile hesapla.
"""
print("\n================= SORU 4 (PythonREPL) =================\n")
ans4 = agent.run(prompt_math)
print("\n================= CEVAP 4 =================\n")
print(ans4)
